In [1]:
#%pip install azure-ai-ml

In [2]:
from azure.ai.ml import MLClient, command, dsl, Input, Output
from azure.ai.ml.entities import RecurrenceTrigger, RecurrencePattern, JobSchedule, Environment, UserIdentityConfiguration
from azure.identity import DefaultAzureCredential

# 1. Authenticate
ml_client = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id="26b57ef9-1628-4837-a014-81f6424512c1",
    resource_group_name="rg-trading-bot-dev",
    workspace_name="ml-trading-workspace"
)

# 2. Define Environment
ml_env = Environment(
    name="trading-cluster-env",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04",
    conda_file="conda.yml"
)

TARGET_COMPUTE = "trading-cluster"

# 3A. Define STEP 1: The ETL Component
etl_component = command(
    code="./",  
    command="python etl_pipeline.py && echo 'done' > ${{outputs.signal_file}}",
    environment=ml_env,
    compute=TARGET_COMPUTE,
    display_name="Step 1: Data Engineering & Clustering",
    name="step_1_etl",
    outputs={"signal_file": Output(type="uri_file")},
    identity=UserIdentityConfiguration()
)

# 3B. Define STEP 2: The AI Component
ai_component = command(
    code="./",  
    command="python ai_agent.py", 
    environment=ml_env,
    compute=TARGET_COMPUTE,
    display_name="Step 2: Agentic Thesis Generation",
    name="step_2_ai_agent",
    inputs={"wait_for_signal": Input(type="uri_file")}, 
    environment_variables={
        "KEY_VAULT_URL": "https://kv-ml-trading-workspace.vault.azure.net/"
    },
    identity=UserIdentityConfiguration()
)

# 4. Construct the Dependency Graph
@dsl.pipeline(
    name="decoupled_quant_pipeline",
    description="2-Step Architecture: ETL followed by AI Generation"
)
def my_quant_pipeline():
    step_1 = etl_component()
    step_2 = ai_component(wait_for_signal=step_1.outputs.signal_file)

pipeline_job = my_quant_pipeline()
pipeline_job.settings.default_compute = TARGET_COMPUTE

# 5. Schedule It
job_schedule = JobSchedule(
    name="daily_market_clustering_schedule",
    trigger=RecurrenceTrigger(
        frequency="day",
        interval=1,
        schedule=RecurrencePattern(hours=21, minutes=30)
    ),
    create_job=pipeline_job
)

ml_client.schedules.begin_create_or_update(job_schedule)
print(f"✅ Secure Pipeline updated using User Identity Pass-Through!")

SyntaxError: '(' was never closed (3094135890.py, line 23)